# SFT From Curated — Close the Loop

After chatting with a fine-tune and curating high-scoring transcripts (see `grade_and_curate_demo.ipynb`), this notebook **trains a new LoRA adapter on the curated examples** — the SFT half of the human-in-the-loop cycle.

Pairs with the rest of the bundled notebooks:

| Stage | Notebook |
|-------|----------|
| Initial RL training | `whitepaper_v1_gsm8k_benchmark.ipynb` / `customer_support_4h.ipynb` / `tool_calling_agent_demo.ipynb` |
| Chat → grade → curate | `grade_and_curate_demo.ipynb` |
| **SFT on curated → next adapter** | **this notebook** |

**Runtime:** ~15 minutes on a Colab A100 for the bundled 2-example smoke set; ~30–90 min for a realistic 100–500 curated examples.

**Open in Colab:** [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stateset/stateset-agents/blob/master/notebooks/sft_from_curated_demo.ipynb)

## 1. Install

In [ ]:
import os
import subprocess

PINNED_COMMIT = '14c0e65'
if not os.path.exists('/content/stateset-agents'):
    subprocess.check_call([
        'git', 'clone', '--quiet',
        'https://github.com/stateset/stateset-agents',
        '/content/stateset-agents'
    ])
subprocess.check_call(['git', '-C', '/content/stateset-agents', 'checkout', '--quiet', PINNED_COMMIT])
%cd /content/stateset-agents
%pip install --quiet -e '.[training]'
%pip install --quiet accelerate bitsandbytes
print('Install complete')

## 2. Stage curated examples

In production this file comes from `make grade-batch ... CURATED=curated.jsonl`. For this demo we stage a small synthetic set.

In [ ]:
import json
from pathlib import Path

curated = Path('/content/curated.jsonl')
examples = [
    {'prompt': 'I need a refund for order #4521', 'response': "I'd be happy to help with your refund. Please confirm the order number and I'll process it right away.", 'score': 0.91, 'source': 's1.jsonl'},
    {'prompt': 'The app keeps crashing on launch', 'response': "I'm sorry to hear that. Could you tell me which version you're on and whether the crash happens consistently?", 'score': 0.88, 'source': 's2.jsonl'},
    {'prompt': 'How do I update my credit card?', 'response': "You can update your card under Settings → Billing → Payment Methods. Let me know if you need help getting there.", 'score': 0.86, 'source': 's3.jsonl'},
    {'prompt': "I was charged twice this month", 'response': "I see — that's frustrating. I'll investigate the duplicate charge. Could you share the dates of both charges?", 'score': 0.92, 'source': 's4.jsonl'},
    {'prompt': 'What are your business hours?', 'response': "We're open Monday–Friday, 9am–6pm Eastern. Happy to help further.", 'score': 0.84, 'source': 's5.jsonl'},
]
curated.write_text('\n'.join(json.dumps(e) for e in examples) + '\n')
print(f'Wrote {len(examples)} curated examples to {curated}')

## 3. Prepare SFT dataset (curated → chat format)

In [ ]:
sft_train = Path('/content/sft_train.jsonl')
result = subprocess.run([
    'python', 'scripts/prepare_sft_dataset.py',
    '--input', str(curated),
    '--format', 'chat',
    '--output', str(sft_train),
    '--min-score', '0.7',
    '--dedup',
    '--stats',
], capture_output=True, text=True, check=True)
print(result.stdout)
print(result.stderr)

## 4. Inspect the prepared dataset

In [ ]:
for i, line in enumerate(sft_train.read_text().splitlines()):
    row = json.loads(line)
    print(f'--- example {i} ---')
    for msg in row['messages']:
        print(f'  [{msg["role"]:>9}] {msg["content"][:80]}…')
    print()

## 5. Run SFT on the prepared data

Two-epoch LoRA r=16 fine-tune. On a Colab A100, ~15 minutes for the bundled 5 examples; longer for realistic dataset sizes.

In [ ]:
import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  device:  {torch.cuda.get_device_name(0)}')
    print(f'  cuda:    {torch.version.cuda}')

In [ ]:
output_dir = Path('/content/outputs/sft_v1')
result = subprocess.run([
    'python', 'scripts/sft_from_curated.py',
    '--dataset', str(sft_train),
    '--base-model', 'Qwen/Qwen3.5-0.8B',
    '--output-dir', str(output_dir),
    '--num-epochs', '2',
    '--lora-r', '16',
    '--per-device-batch-size', '2',
    '--gradient-accumulation-steps', '4',
    '--max-length', '512',
], check=False)
print(f'\nExit code: {result.returncode}')
if output_dir.exists():
    print(f'\nAdapter contents:')
    for p in sorted(output_dir.iterdir()):
        print(f'  {p.name}')

## 6. Chat with the fine-tuned adapter

Sanity-check the result via the in-process chat path (same one `stateset-agents chat` uses).

In [ ]:
if not output_dir.exists() or not torch.cuda.is_available():
    print('Skipping — no trained adapter to chat with.')
else:
    import asyncio
    from stateset_agents.core.agent import MultiTurnAgent
    from stateset_agents.core.agent_config import AgentConfig

    agent = MultiTurnAgent(AgentConfig(
        model_name='Qwen/Qwen3.5-0.8B',
        peft_path=str(output_dir),
        max_new_tokens=200,
        temperature=0.0,
        do_sample=False,
        torch_dtype='bfloat16',
    ))
    await agent.initialize()

    for prompt in [
        'I want a refund for my recent purchase',
        'My account is showing the wrong balance',
        'Are you guys open on weekends?',
    ]:
        print(f'\nUser: {prompt}')
        response = await agent.generate_response(prompt)
        print(f'Agent: {response[:300]}')

## 7. What's next

You now have:

- A trained LoRA adapter under `/content/outputs/sft_v1/`
- It loads via `AgentConfig.peft_path`, so it composes with the rest of the framework:
  - **Serve it:** `stateset-agents serve --checkpoint /content/outputs/sft_v1 --base-model Qwen/Qwen3.5-0.8B`
  - **Benchmark it:** drop a `peft_path` into your `GSPOConfig` and run another RL pass
  - **Chat with it:** `stateset-agents chat --model Qwen/Qwen3.5-0.8B --checkpoint outputs/sft_v1`
  - **Capture more transcripts → curate → SFT again** — each round, your model and your reward function tighten on each other

This is the loop closure that makes consulting fine-tunes a continuous improvement process rather than a one-shot deliverable. See [`docs/PLATFORM_TOUR.md`](../docs/PLATFORM_TOUR.md) for the full developer journey.